# ME 144/244: Modeling, Simulation, and Digital Twins of Drone-Based Systems 
## Project 3: Swarm
## Spring 2026 Semester
### GSI - Tommy Hosmer

## Notebook to generate figures

In [ ]:
import pickle
from simulation import Simulation
from animation import Animation
from swarmga import SwarmGA

In [ ]:
import importlib
import pickle
import write_parameters as wp

# Ensure parameters.pkl reflects the latest write_parameters.py edits
wp = importlib.reload(wp)

with open('parameters.pkl', 'wb') as f:
    pickle.dump(wp.parameters_dict, f)


## Load the parameter values that define the GA and Simulation

In [ ]:
with open('parameters.pkl', 'rb') as file:
    parameters = pickle.load(file)

# Run GA

In [ ]:
swarmGA = SwarmGA(parameters)
swarmGA.initialize_special_params()
# Return dictionary of results
data = swarmGA.specialGA(print_statements=True)

## Dump GA results into a pickle file

### This allows you to reuse data from the GA in the event you restart the notebook. With the data saved you can simply read the pkl and run the animations/plots when convenient

In [ ]:
with open('ga_results.pkl', 'wb') as f:
        pickle.dump(data, f)

# Run Simulation with GA Results


In [ ]:
# Run this cell in case of kernel restart and need GA data from pickle file
with open('ga_results.pkl', 'rb') as file:
    data = pickle.load(file)

In [ ]:
LAM = data['best_p_strings'][0]
sim = Simulation(parameters=parameters, path_to_genes='')
PI, posData, tarData, obsData, counter, Mstar, Tstar, Lstar =  sim.run_simulation(True, LAM)

# Make Plots and Animations

In [ ]:
animationObject = Animation(counter, tarData, posData, obsData, parameters)

# Question 1

In [ ]:
animationObject.plot_costs_over_generations('ga_results.pkl','figures')

## Question 2

In [ ]:
animationObject.plot_star_vals('ga_results.pkl','figures')

## Question 3

In [ ]:
for i in range(4):
    res = []
    LAM = data['best_p_strings'][i]
    for j in LAM:
        res.append(float(round(j,3)))
    print(f'Lambda[{i}]: {res}')

    sim = Simulation(parameters=parameters, path_to_genes='')
    PI, _, _, _, _, _, _, _ =  sim.run_simulation(True, LAM)
    print(f'PI[{i}]: {float(round(PI,3))}\n')

## Question 5

In [ ]:
# Save this animation, then take screenshots
# Save MP4 into figures/ for turn-in
animationObject.animation('figures')


In [ ]:
# Export 5 evenly spaced frames from the MP4 into figures/
import importlib
from pathlib import Path

import export_frames as ef

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

# Reload in case export_frames.py was edited
ef = importlib.reload(ef)

# The animation saves as "{Nm}_agents_{No}_obs_{Nt}_tar.mp4"
mp4_name = f"{parameters['Nm']}_agents_{parameters['No']}_obs_{parameters['Nt']}_tar.mp4"
mp4_path = Path(mp4_name)

# If the MP4 is already in figures/, use that; otherwise use the one in cwd
if not mp4_path.exists() and (fig_dir / mp4_name).exists():
    mp4_path = fig_dir / mp4_name

pngs = ef.export_evenly_spaced_frames(mp4_path=mp4_path, out_dir=fig_dir, n_frames=5, prefix='snapshot')
print('Wrote snapshots:')
for p in pngs:
    print(' ', p)


In [ ]:
# Debug: confirm where snapshots are being written
from pathlib import Path

print('cwd:', Path.cwd())
for name in ['snapshot_01.png', 'snapshot_05.png']:
    p = Path('figures') / name
    print(name, '->', p.resolve(), 'exists=', p.exists())
